# Tanush — Week 1 ECG benchmark pipeline

This notebook uses the team's shared storage contract: canonical leads `I, II, III, aVR, aVL, aVF, V1–V6`, 500 Hz, 10 seconds, millivolts, `float32`, shape `(12, 5000)`. It can reproduce Georgia preprocessing in Drive and verifies that the ECG-FM pretrained checkpoint loads with `fairseq-signals`.

In [ ]:
from pathlib import Path

# The checkpoint test is self-contained and does not require Drive mounting.
# Change this to True only when intentionally rebuilding Georgia in Drive.
RUN_GEORGIA_PREPROCESSING = False
DRIVE_ROOT = Path('/content')
PIPELINE_ROOT = None
PROCESSED_ROOT = DRIVE_ROOT / 'processed'
print({'checkpoint_workspace': str(DRIVE_ROOT), 'rebuild_georgia': RUN_GEORGIA_PREPROCESSING})

## Reproduce Georgia preprocessing

The raw download stays on temporary Colab storage. Only the contract-compliant NumPy signals, index, QC report, and five-sample plot are written to the shared Drive.

In [ ]:
if RUN_GEORGIA_PREPROCESSING:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', 'scipy>=1.11', 'matplotlib>=3.8'])
    import kagglehub
    candidates = [
        Path('/content/drive/MyDrive/LSTS/ecg_benchmark_1/week1_tanush'),
        Path('/content/drive/MyDrive/ecg_benchmark_1/week1_tanush'),
        Path('/content/drive/MyDrive/ecg_benchmark/week1_tanush'),
    ]
    PIPELINE_ROOT = next((path for path in candidates if path.exists()), None)
    assert PIPELINE_ROOT is not None, 'Could not find the shared week1_tanush folder'
    PROCESSED_ROOT = Path('/content/drive/MyDrive/LSTS/ecg_benchmark/processed')
    PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
    GEORGIA_RAW = Path(kagglehub.dataset_download(
        'physionet/georgia-12lead-ecg-challenge-database',
        output_dir='/content/raw/georgia',
    )) / 'Georgia'
    assert GEORGIA_RAW.exists(), f'Missing extracted Georgia folder: {GEORGIA_RAW}'
    print('Georgia source:', GEORGIA_RAW)
else:
    print('Georgia rebuild skipped: the verified 10,344-record output is already in Drive.')

In [ ]:
if RUN_GEORGIA_PREPROCESSING:
    import os, sys
    os.chdir(PIPELINE_ROOT)
    sys.path.insert(0, str(PIPELINE_ROOT))
    !python src/data/preprocess_georgia.py --input-root "{GEORGIA_RAW}" --output-root "{PROCESSED_ROOT}/georgia" --mapping data/label_mappings/physionet_challenge_2020_five_labels.csv
    !python src/data/sanity_check.py --signal-dir "{PROCESSED_ROOT}/georgia/signals" --report "{PROCESSED_ROOT}/georgia/georgia_sanity.csv" --plot "{PROCESSED_ROOT}/georgia/georgia_five_sample_plot.png" --seed 42
else:
    print('Georgia preprocessing skipped.')

## Load and verify the ECG-FM pretrained checkpoint

ECG-FM is not a Hugging Face `transformers` model. The weights are downloaded from Hugging Face, then loaded with the official `fairseq-signals` code used by ECG-FM.

In [ ]:
import os, sys, subprocess
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
repo = Path('/content/fairseq-signals')
if not repo.exists():
    subprocess.check_call(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Jwoo5/fairseq-signals.git', str(repo)])
install_env = dict(os.environ, MAX_JOBS='2')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)], env=install_env)
sys.path.insert(0, str(repo))
import fairseq_signals
print({'fairseq_signals_import': 'PASS', 'source': fairseq_signals.__file__})

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_dir = DRIVE_ROOT / 'checkpoints' / 'ecg-fm'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = hf_hub_download(
    repo_id='wanglab/ecg-fm',
    filename='mimic_iv_ecg_physionet_pretrained.pt',
    local_dir=checkpoint_dir,
)
print('Checkpoint:', checkpoint_path)

In [ ]:
import torch
from fairseq_signals.models import build_model_from_checkpoint

model = build_model_from_checkpoint(checkpoint_path=checkpoint_path)
model.eval()
parameter_count = sum(parameter.numel() for parameter in model.parameters())
assert parameter_count > 90_000_000
print({
    'status': 'PASS',
    'model_class': type(model).__name__,
    'parameters': parameter_count,
    'checkpoint': Path(checkpoint_path).name,
})

In [ ]:
import numpy as np

LEADS = ('I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6')

def shared_signal_to_ecg_fm_windows(signal, valid_num_samples=5000):
    """Apply ECG-FM's official Standardize-then-segment input transform."""
    signal = np.asarray(signal, dtype=np.float32)
    if signal.shape != (12, 5000):
        raise ValueError(f'Expected (12, 5000), got {signal.shape}')
    if not np.isfinite(signal).all():
        raise ValueError('Signal contains NaN or infinity')
    if valid_num_samples not in (2500, 5000):
        raise ValueError(f'Expected 2500 or 5000 valid samples, got {valid_num_samples}')
    valid_signal = signal[:, :valid_num_samples]
    means = valid_signal.mean(axis=1, keepdims=True)
    standard_deviations = valid_signal.std(axis=1, keepdims=True)
    standardized = np.divide(
        valid_signal - means,
        standard_deviations + 1e-8,
        out=np.zeros_like(valid_signal),
        where=standard_deviations > 0,
    )
    windows = standardized.reshape(12, -1, 2500).transpose(1, 0, 2)
    return torch.from_numpy(windows)

example = np.random.default_rng(42).normal(size=(12, 5000)).astype(np.float32)
model_input = shared_signal_to_ecg_fm_windows(example)
assert model_input.shape == (2, 12, 2500)
print({'adapter_status': 'PASS', 'model_input_shape': tuple(model_input.shape)})

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    model = model.to(device)
    with torch.inference_mode():
        features = model.extract_features(model_input[:1].to(device), padding_mask=None, mask=False)
    print({'forward_status': 'PASS', 'encoded_shape': tuple(features['x'].shape)})
else:
    print('Checkpoint load passed. GPU unavailable, so the optional forward smoke test was skipped.')